In [1]:
from pathlib import Path
import sys
import os
import optuna

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from frost_evaporator import MultiExperimentAnalyzer, FrostEvaporatorSimulation, SimulationVisualizer

###########################################################################################
# Initialisation
###########################################################################################

cutoff_pct = 0.05

# --- Abt Setup (7.0 mm) ---
analyzer_abt = MultiExperimentAnalyzer(experiment_type='OptiAbt')
sim_abt = FrostEvaporatorSimulation("config_OptiAbt.yaml")
viz_abt = SimulationVisualizer(sim_abt.params)
path_exp_abt = Path(r"D:\mbc_nba\OptiAbt_Daten\Versuche\CSV_Export")
experiment_groups_abt = {
    "Abt-a": [56],
    "Abt-b": [40],
    # "Abt-c": [28],
    # "Abt-d": [13],
}
cached_abt_data = {}
for name, ids in experiment_groups_abt.items():
    cached_abt_data[name] = analyzer_abt.analyze(exp_ids=ids, data_path=path_exp_abt, cutoff_pct=cutoff_pct, time_step=sim_abt.params.time_step)

abt_weights = {
    "Abt-a": 1.0,
    "Abt-b": 1.0, 
    "Abt-c": 1.0, 
    "Abt-d": 1.0
}

# --- Horst Setup (1.75 mm) ---
analyzer_horst = MultiExperimentAnalyzer(experiment_type='OptiHorst')
sim_horst      = FrostEvaporatorSimulation("config_OptiHorst.yaml")
viz_horst      = SimulationVisualizer(sim_horst.params)
path_exp_horst = Path(r"D:\mbc_nba\OptiHorst\Daten\Abtauen\CSV_Export")
experiment_groups_horst = {
    "Horst-a": [96],
    "Horst-b": [25],
    "Horst-c": [45],
    "Horst-d": [102],
    "Horst-e": [60],
}
cached_horst_data = {}
for name, ids in experiment_groups_horst.items():
    cached_horst_data[name] = analyzer_horst.analyze(exp_ids=ids, data_path=path_exp_horst, cutoff_pct=cutoff_pct, time_step=sim_horst.params.time_step)

horst_weights = {
    "Horst-a": 1.0, 
    "Horst-b": 1.0, 
    "Horst-c": 1.0, 
    "Horst-d": 1.0, 
    "Horst-e": 1.0,
}

###########################################################################################
# The Objective Function
###########################################################################################

def objective(trial):
    # # 1. Suggest your variables
    # betta_horst = trial.suggest_float('betta_1_75mm', 0.35, 0.5) 
    # betta_abt = trial.suggest_float('betta_7_0mm', 0.7, 1.4)    
    
    # gap_horst = 0.00175
    # gap_abt = 0.007
    
    # slope = (betta_abt - betta_horst) / (gap_abt - gap_horst)
    # intercept = betta_abt - (slope * gap_abt)
    
    # if (slope * 0.0003 + intercept) < 0.0:
    #     raise optuna.exceptions.TrialPruned()

    # Store suggested values in standard variables first
    # h_conv_air_val      = trial.suggest_float('h_conv_air', 0.8, 1.2)
    # surface_density_val = trial.suggest_float('surface_density', 0.6, 1.3)
    # k_frost_val         = trial.suggest_float('k_frost', 0.7, 1.3)
    # frost_diffusion_val = trial.suggest_float('frost_diffusion', 0.3, 3.0)
    pressure_loss_val   = trial.suggest_float('pressure_loss', 0.65, 0.8)
    # roughness_C_val     = trial.suggest_float('roughness_Cf', 10, 70)
    # roughness_n_val     = trial.suggest_float('roughness_n', 0.5, 4.0)

    # 2. Assign accepted factors for the simulation
    # new_factors = {
    #     'h_conv_air':          h_conv_air_val, 
    #     'betta_intercept_air': intercept,
    #     'betta_slope_air':     slope,
    #     'surface_density':     surface_density_val, 
    #     'k_frost':             k_frost_val,
    #     'frost_diffusion':     frost_diffusion_val, 
    #     'pressure_loss':       pressure_loss_val, 
    #     'roughness_C':         roughness_C_val,
    #     'roughness_n':         roughness_n_val,
    # }

    new_factors = {
        'h_conv_air':          1.19, 
        'betta_intercept_air': 0.103,
        'betta_slope_air':     146.67,
        'surface_density':     1.02, 
        'k_frost':             0.80,
        'frost_diffusion':     2.59, 
        'pressure_loss':       pressure_loss_val, # 0.73 
        'roughness_C':         8.24,
        'roughness_n':         2.62,
    }


    print(new_factors)


    new_model_choices = {
        'frost_density_choice':      'hayashi_1977',
        'frost_conductivity_choice': 'yonko_sepsy_1967',
        'h_conv_air_choice':         'Wang',
    }

    total_summed_error = 0.0

    # =========================================================
    # Do OptiAbt Simulations (7.0 mm)
    # =========================================================
    try:
        sim_abt.update_correction_factors(new_factors, new_model_choices)
        
        for name, exp_data in cached_abt_data.items():
            states, inputs = sim_abt.run_validation(exp_data)
            exp_id = experiment_groups_abt[name][0] 

            use_melted_mass = (name == "Abt-a")
            
            df_err = viz_abt.get_relative_error_table(
                states, inputs, [exp_id], path_exp_abt, cutoff_pct, experiment_name="OptiAbt", use_melted_mass=use_melted_mass
            )

            row = df_err.loc[exp_id].fillna(0.0)
            print(row)

            # Only a and b have data for dp
            err_dp = row["dp"] if name in ["Abt-a", "Abt-b"] else 0.0
            
            # Sum of Squares of relative errors
            err_val = (err_dp**2) # (row["m_frost"]**2) + (err_dp**2)  + (row["Q"]**2)
            total_summed_error += err_val * abt_weights.get(name, 1.0) 
            
    except Exception as e:
        print(f"Trial {trial.number} failed in Abt: {e}")
        raise optuna.exceptions.TrialPruned()


    # =========================================================
    # Do OptiHorst Simulations (1.75 mm)
    # =========================================================
    try:
        sim_horst.update_correction_factors(new_factors, new_model_choices)
        
        for name, exp_data in cached_horst_data.items():
            states, inputs = sim_horst.run_validation(exp_data)
            exp_id = experiment_groups_horst[name][0]
            
            df_err = viz_horst.get_relative_error_table(
                states, inputs, [exp_id], path_exp_horst, cutoff_pct, experiment_name="OptiHorst"
            )
            
            row = df_err.loc[exp_id].fillna(0.0)
            print(row)
            
            # Sum of Squares of relative errors
            err_val =  (row["dp"]**2) # (row["m_frost"]**2) + (row["dp"]**2)  + (row["Q"]**2)
            total_summed_error += err_val * horst_weights.get(name, 1.0)
            
    except Exception as e:
        print(f"Trial {trial.number} failed in Horst: {e}")
        raise optuna.exceptions.TrialPruned()


    # 4. Return the combined error
    error = total_summed_error
    error = min(error, 1e5)

    return error
###########################################################################################
# Optimization Execution
###########################################################################################

storage_name = "sqlite:///evaporator_optimization.db"

study = optuna.create_study(
    study_name="evaporator_tuning_pressure_loss_all", 
    storage=storage_name,
    load_if_exists=True,
    direction="minimize"
)

study.optimize(objective, n_trials=20)

print(f"Best factors: {study.best_params}")

# D:
# cd D:\mbc_nba\vclibpy\frost_evaporator_nba\notebooks
# optuna-dashboard sqlite:///evaporator_optimization.db

--- Aggregating Data (OptiAbt) for Experiments: [56] ---
--- Aggregating Data (OptiAbt) for Experiments: [40] ---
--- Aggregating Data (OptiHorst) for Experiments: [96] ---
--- Aggregating Data (OptiHorst) for Experiments: [25] ---
--- Aggregating Data (OptiHorst) for Experiments: [45] ---
--- Aggregating Data (OptiHorst) for Experiments: [102] ---
--- Aggregating Data (OptiHorst) for Experiments: [60] ---


[I 2026-03-12 14:02:15,876] A new study created in RDB with name: evaporator_tuning_pressure_loss_all


{'h_conv_air': 1.19, 'betta_intercept_air': 0.103, 'betta_slope_air': 146.67, 'surface_density': 1.02, 'k_frost': 0.8, 'frost_diffusion': 2.59, 'pressure_loss': 0.7206423975489802, 'roughness_C': 8.24, 'roughness_n': 2.62}
--- Simulation of Experiment Combined_1_OptiAbt ---
--- Starting Simulation ---


Sim Combined_1_OptiAbt: 100%|██████████| 53/53 [01:09<00:00,  1.32s/it, Frost Surface Temp > 0°C in Layers 1, 2, 3, 4]


dp           26.037675
h_ref_out     1.330622
Q             3.836320
m_frost       5.353303
Name: 56, dtype: float64
--- Simulation of Experiment Combined_1_OptiAbt ---
--- Starting Simulation ---


...

dp           16.294060
h_ref_out     0.832858
Q             1.943281
m_frost      14.292772
Name: 45, dtype: float64
--- Simulation of Experiment Combined_1_OptiHorst ---
--- Starting Simulation ---


Sim Combined_1_OptiHorst: 100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


dp           11.929509
h_ref_out     1.630815
Q             3.800601
m_frost      59.286225
Name: 102, dtype: float64
--- Simulation of Experiment Combined_1_OptiHorst ---
--- Starting Simulation ---


Sim Combined_1_OptiHorst: 100%|██████████| 33/33 [00:22<00:00,  1.48it/s]
[I 2026-03-12 15:06:53,081] Trial 19 finished with value: 3634.569085480225 and parameters: {'pressure_loss': 0.653060753435881}. Best is trial 18 with value: 2383.450400991639.


dp           30.129186
h_ref_out     0.634279
Q             1.656180
m_frost      11.533307
Name: 60, dtype: float64
Best factors: {'pressure_loss': 0.7998278136735343}
